In [2]:
import os, time, random, numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------- Data Loading --------------------
def load_dataset(data_path, dataset_name, config):
    file_path = os.path.join(data_path, config['file'])
    data = sio.loadmat(file_path)[config['data_key']]
    if config['file'] == config['gt_file']:
        labels = sio.loadmat(file_path)[config['gt_key']]
    else:
        gt_path = os.path.join(data_path, config['gt_file'])
        labels = sio.loadmat(gt_path)[config['gt_key']]
    if len(labels.shape) == 3:
        labels = labels.squeeze()
    return data, labels

def applyPCA(X, numComponents):
    original_shape = X.shape
    X_reshaped = X.reshape(-1, X.shape[-1])
    pca = PCA(n_components=min(numComponents, X.shape[-1]), whiten=True)
    X_pca = pca.fit_transform(X_reshaped)
    return X_pca.reshape(original_shape[0], original_shape[1], -1)

def createImageCubes(X, y, windowSize=5):
    margin = windowSize // 2
    paddedX = np.pad(X, ((margin, margin), (margin, margin), (0,0)), mode='reflect')
    patches, labels_ = [], []
    for r in range(margin, paddedX.shape[0]-margin):
        for c in range(margin, paddedX.shape[1]-margin):
            if y[r-margin, c-margin] > 0:
                patch = paddedX[r-margin:r+margin+1, c-margin:c+margin+1]
                patches.append(patch)
                labels_.append(y[r-margin, c-margin]-1)
    return np.array(patches), np.array(labels_)

# -------------------- Adaptive Clustering (unchanged) --------------------
class AdaptiveClustering(nn.Module):
    def __init__(self, weight_shape):
        super().__init__()
        self.out_channels = weight_shape[0]
        self.D = int(np.prod(weight_shape[1:]))
        self.keep_ratio = nn.Parameter(torch.tensor(0.5))
        self.tau = nn.Parameter(torch.tensor(1.0))

    @property
    def K(self):
        k = int(max(2, torch.sigmoid(self.keep_ratio).item() * self.out_channels))
        return min(k, self.out_channels)

    def get_centroids(self):
        if not hasattr(self, '_centroids_buffer') or self._centroids_buffer.shape[0] != self.K:
            self.register_buffer('_centroids_buffer', torch.randn(self.K, self.D) * 0.01)
        return self._centroids_buffer

    def forward(self, W):
        W_flat = W.view(W.shape[0], -1)
        O, D = W_flat.shape
        K_eff = self.K
        centroids = self.get_centroids()[:K_eff]
        tau_val = self.tau.abs() + 0.05
        dist = torch.cdist(W_flat, centroids)
        A = F.softmax(-dist / tau_val, dim=1)
        W_compressed = torch.mm(A, centroids)
        return W_compressed.view_as(W), A

# -------------------- True SpectralGPT Block (ViT with 3D tokens) --------------------
class SpectralGPTBlock(nn.Module):
    """Transformer block matching SpectralGPT (ViT) architecture."""
    def __init__(self, embed_dim, num_heads, mlp_ratio, layer_idx, compress_layers, gate_init):
        super().__init__()
        self.layer_idx = layer_idx
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.compress_layers = compress_layers

        # Pre-norm
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Attention (separate Q/K/V for compression compatibility)
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=True)

        # MLP (ratio 4, as in ViT/SpectralGPT)
        hidden = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.fc2 = nn.Linear(hidden, embed_dim)

        self.compression_module_names = [
            f'q_{layer_idx}', f'k_{layer_idx}', f'v_{layer_idx}',
            f'attn_out_{layer_idx}', f'fc1_{layer_idx}', f'fc2_{layer_idx}'
        ]

    def register_compression_modules(self, compression_modules, gates):
        for name in self.compression_module_names:
            if name in self.compress_layers:
                if name.startswith('q_'):
                    weight = self.q_proj.weight
                elif name.startswith('k_'):
                    weight = self.k_proj.weight
                elif name.startswith('v_'):
                    weight = self.v_proj.weight
                elif name.startswith('attn_out_'):
                    weight = self.out_proj.weight
                elif name.startswith('fc1_'):
                    weight = self.fc1.weight
                elif name.startswith('fc2_'):
                    weight = self.fc2.weight
                else:
                    continue
                compression_modules[name] = AdaptiveClustering(weight.shape)
                gates[name] = nn.Parameter(torch.tensor(0.0))

    def get_compressed_weight(self, model, name, original_weight):
        if name not in model.compression_modules:
            return original_weight
        gate = model.get_gate(name)
        if gate < 0.01:
            return original_weight
        compressed, _ = model.compression_modules[name](original_weight)
        return (1 - gate) * original_weight + gate * compressed

    def forward(self, x, model):
        B, N, D = x.shape
        H = self.num_heads
        head_dim = D // H

        # --- Attention (pre-norm) ---
        x_norm = self.norm1(x)
        q_w = self.get_compressed_weight(model, f'q_{self.layer_idx}', self.q_proj.weight)
        k_w = self.get_compressed_weight(model, f'k_{self.layer_idx}', self.k_proj.weight)
        v_w = self.get_compressed_weight(model, f'v_{self.layer_idx}', self.v_proj.weight)
        out_w = self.get_compressed_weight(model, f'attn_out_{self.layer_idx}', self.out_proj.weight)

        q = F.linear(x_norm, q_w, self.q_proj.bias)
        k = F.linear(x_norm, k_w, self.k_proj.bias)
        v = F.linear(x_norm, v_w, self.v_proj.bias)

        q = q.view(B, N, H, head_dim).transpose(1, 2)
        k = k.view(B, N, H, head_dim).transpose(1, 2)
        v = v.view(B, N, H, head_dim).transpose(1, 2)

        attn = F.scaled_dot_product_attention(q, k, v)
        attn = attn.transpose(1, 2).reshape(B, N, D)
        attn = F.linear(attn, out_w, self.out_proj.bias)
        x = x + attn

        # --- MLP (pre-norm) ---
        x_norm2 = self.norm2(x)
        ffn_w1 = self.get_compressed_weight(model, f'fc1_{self.layer_idx}', self.fc1.weight)
        ffn_w2 = self.get_compressed_weight(model, f'fc2_{self.layer_idx}', self.fc2.weight)
        ffn = F.linear(x_norm2, ffn_w1, self.fc1.bias)
        ffn = F.gelu(ffn)
        ffn = F.linear(ffn, ffn_w2, self.fc2.bias)
        x = x + ffn
        return x

# -------------------- True SpectralGPT Foundation Model --------------------
class LargeCompressibleSpectralGPT(nn.Module):
    """True SpectralGPT: 3D cube tokenization, MAE-style 90% masking, encoder-decoder."""
    def __init__(self, n_classes, S, L, patch_size=5, spectral_group=3,
                 embed_dim=768, num_heads=12, num_layers=12,
                 decoder_embed_dim=512, decoder_num_heads=8, decoder_depth=4,
                 mlp_ratio=4.0, compress_layers=None, gate_init=0.0):
        super().__init__()
        self.compress_layers = compress_layers if compress_layers is not None else []
        self.compression_modules = nn.ModuleDict()
        self.gates = nn.ParameterDict()
        self.embed_dim = embed_dim
        self.num_layers = num_layers
        self.S = S
        self.L = L
        self.patch_size = patch_size
        self.spectral_group = spectral_group

        # 3D cube tokenization: each token = p × p × k
        self.num_patches_h = S // patch_size
        self.num_patches_w = S // patch_size
        self.num_spectral_groups = L // spectral_group
        self.num_tokens = self.num_patches_h * self.num_patches_w * self.num_spectral_groups

        # 3D patch embedding: (p*p*k) → embed_dim
        self.patch_embed = nn.Linear(patch_size * patch_size * spectral_group, embed_dim)

        # Positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Encoder blocks
        self.blocks = nn.ModuleList()
        for i in range(num_layers):
            self.blocks.append(
                SpectralGPTBlock(embed_dim, num_heads, mlp_ratio, i, self.compress_layers, gate_init)
            )

        self.norm = nn.LayerNorm(embed_dim)

        # Decoder (shallow, for MAE pretraining)
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, decoder_embed_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)

        self.decoder_blocks = nn.ModuleList()
        for _ in range(decoder_depth):
            self.decoder_blocks.append(
                SpectralGPTBlock(decoder_embed_dim, decoder_num_heads, mlp_ratio, 0,
                                 [], 0.0)
            )

        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size * patch_size * spectral_group)

        # Classification head: average pooling + one-layer MLP
        self.head = nn.Linear(embed_dim, n_classes)

        self._register_compression_modules()

    def _register_compression_modules(self):
        if 'patch_embed' in self.compress_layers:
            self.compression_modules['patch_embed'] = AdaptiveClustering(self.patch_embed.weight.shape)
            self.gates['patch_embed'] = nn.Parameter(torch.tensor(0.0))
        for block in self.blocks:
            block.register_compression_modules(self.compression_modules, self.gates)

    def get_gate(self, name):
        if self.gates is None or name not in self.gates:
            return 0.0
        return torch.sigmoid(self.gates[name])

    def get_compressed_weight(self, name, original_weight):
        if self.compression_modules is None or name not in self.compression_modules:
            return original_weight
        gate = self.get_gate(name)
        if gate < 0.01:
            return original_weight
        compressed, _ = self.compression_modules[name](original_weight)
        return (1 - gate) * original_weight + gate * compressed

    def patchify(self, x):
        """x: (B, S, S, L) → (B, num_tokens, p*p*k)"""
        B = x.shape[0]
        p, k = self.patch_size, self.spectral_group
        x = x.view(B, self.num_patches_h, p, self.num_patches_w, p, self.num_spectral_groups, k)
        x = x.permute(0, 1, 3, 5, 2, 4, 6).contiguous()
        x = x.view(B, self.num_tokens, p * p * k)
        return x

    def random_masking(self, x, mask_ratio):
        """MAE-style random masking. x: (B, N, D) → (B, N_vis, D), mask, ids_restore"""
        B, N, D = x.shape
        len_keep = int(N * (1 - mask_ratio))
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        mask = torch.ones(B, N, device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio=0.9):
        """Encode visible tokens only."""
        B = x.shape[0]
        x = self.patchify(x)                               # (B, N, p*p*k)
        w_embed = self.get_compressed_weight('patch_embed', self.patch_embed.weight)
        x = F.linear(x, w_embed, self.patch_embed.bias)    # (B, N, embed_dim)
        x = x + self.pos_embed
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        for block in self.blocks:
            x = block(x, self)
        x = self.norm(x)
        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        """Decode and reconstruct masked tokens."""
        x = self.decoder_embed(x)
        B, N_vis, _ = x.shape
        mask_tokens = self.mask_token.repeat(B, self.num_tokens - N_vis, 1)
        x_ = torch.cat([x, mask_tokens], dim=1)
        x_ = torch.gather(x_, dim=1,
                          index=ids_restore.unsqueeze(-1).repeat(1, 1, x_.shape[2]))
        x_ = x_ + self.decoder_pos_embed
        for block in self.decoder_blocks:
            x_ = block(x_, self)
        x_ = self.decoder_norm(x_)
        x_ = self.decoder_pred(x_)
        return x_

    def forward(self, x, mask_ratio=0.0):
        """
        If mask_ratio > 0: MAE pretraining forward (returns reconstruction + mask).
        If mask_ratio == 0: classification forward (returns logits).
        """
        if mask_ratio > 0:
            latent, mask, ids_restore = self.forward_encoder(x, mask_ratio)
            pred = self.forward_decoder(latent, ids_restore)
            return pred, mask
        else:
            # Classification: full encoder + average pooling + MLP head
            B = x.shape[0]
            x = self.patchify(x)
            w_embed = self.get_compressed_weight('patch_embed', self.patch_embed.weight)
            x = F.linear(x, w_embed, self.patch_embed.bias)
            x = x + self.pos_embed
            for block in self.blocks:
                x = block(x, self)
            x = self.norm(x)
            # Average pooling along token dimension (SpectralGPT classification head)
            x = x.mean(dim=1)                              # (B, embed_dim)
            return self.head(x)

    def sparsity_loss(self, lambda_base=3.0):
        if self.gates is None or len(self.gates) == 0:
            return torch.tensor(0.0)
        loss = 0.0
        for name, gate_param in self.gates.items():
            gate = self.get_gate(name)
            loss += lambda_base * (1 - gate)
            if name in self.compression_modules:
                keep = torch.sigmoid(self.compression_modules[name].keep_ratio)
                loss += 1.5 * keep
        return loss / len(self.gates)

    def get_learned_K(self):
        return {name: mod.K for name, mod in self.compression_modules.items()}

# -------------------- Training (two-stage, unchanged) --------------------
def train_two_stage(model, train_loader, X_test_t, y_test_t,
                    epochs_stage1=150, epochs_stage2=100, final_lambda=3.0, lr=0.0005):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    ce_loss = nn.CrossEntropyLoss()
    print("\n--- Stage 1: Training without compression loss ---")
    best_val_acc = 0
    best_state = None
    patience = 30
    no_improve = 0
    for epoch in range(epochs_stage1):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)                                 # classification mode
            loss = ce_loss(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            out_val = model(X_test_t)
            val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience and best_val_acc > 0.95:
                print(f"Early stopping at epoch {epoch+1} (best val_acc={best_val_acc:.4f})")
                model.load_state_dict(best_state)
                break
        if (epoch+1) % 20 == 0:
            print(f"Stage1 Epoch {epoch+1}/{epochs_stage1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    print("\n--- Stage 2: Adding compression loss (K evolution) ---")
    for epoch in range(epochs_stage2):
        model.train()
        total_loss = 0.0
        lam = final_lambda * (epoch / max(1, epochs_stage2-1))
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss += lam * model.sparsity_loss()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch+1) % 10 == 0 or epoch == epochs_stage2-1:
            model.eval()
            with torch.no_grad():
                out_val = model(X_test_t)
                val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
            gates = {name: model.get_gate(name).item() for name in model.gates}
            k_vals = model.get_learned_K()
            print(f"\n📊 Epoch {epoch+1}/{epochs_stage2} | λ={lam:.3f} | Val Acc={val_acc:.4f}")
            print(f"   Gates: {gates}")
            print(f"   Learned K: {k_vals}")
    return model

def evaluate_full(model, X, y_true):
    model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        out = model(X)
    t2 = time.perf_counter()
    pred = out.argmax(dim=1).cpu().numpy()
    y_true = y_true.cpu().numpy()
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, pred, average='macro', zero_division=0),
        'inf_time_ms': (t2 - t1) * 1000
    }

# -------------------- Build DKM Compressed Tiny SpectralGPT --------------------
def build_dkm_tiny_model(original_model, learned_K, input_dim, n_classes,
                         num_layers=4, patch_size=5, spectral_group=3):
    new_embed_dim = learned_K.get('patch_embed', original_model.embed_dim)
    S = original_model.S
    L = original_model.L
    num_tokens = original_model.num_tokens

    class DKMCompressedSpectralGPT(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed_dim = new_embed_dim
            self.num_tokens = num_tokens
            self.patch_size = patch_size
            self.spectral_group = spectral_group
            self.num_patches_h = S // patch_size
            self.num_patches_w = S // patch_size
            self.num_spectral_groups = L // spectral_group

            self.patch_embed = nn.Linear(patch_size * patch_size * spectral_group, new_embed_dim)
            self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, new_embed_dim))
            self.layers = nn.ModuleList()
            for i in range(num_layers):
                qk_dim = min(
                    learned_K.get(f'q_{i}', new_embed_dim),
                    learned_K.get(f'k_{i}', new_embed_dim),
                    learned_K.get(f'v_{i}', new_embed_dim),
                )
                ffn1_dim = learned_K.get(f'fc1_{i}', new_embed_dim * 4)
                self.layers.append(nn.ModuleDict({
                    'q': nn.Linear(new_embed_dim, qk_dim),
                    'k': nn.Linear(new_embed_dim, qk_dim),
                    'v': nn.Linear(new_embed_dim, qk_dim),
                    'out': nn.Linear(qk_dim, new_embed_dim),
                    'fc1': nn.Linear(new_embed_dim, ffn1_dim),
                    'fc2': nn.Linear(ffn1_dim, new_embed_dim),
                    'norm1': nn.LayerNorm(new_embed_dim),
                    'norm2': nn.LayerNorm(new_embed_dim)
                }))
            self.norm = nn.LayerNorm(new_embed_dim)
            self.head = nn.Linear(new_embed_dim, n_classes)

        def patchify(self, x):
            B = x.shape[0]
            p, k = self.patch_size, self.spectral_group
            x = x.view(B, self.num_patches_h, p, self.num_patches_w, p, self.num_spectral_groups, k)
            x = x.permute(0, 1, 3, 5, 2, 4, 6).contiguous()
            return x.view(B, self.num_tokens, p * p * k)

        def forward(self, x):
            B = x.shape[0]
            x = self.patchify(x)
            x = self.patch_embed(x) + self.pos_embed
            for layer in self.layers:
                x_norm = layer['norm1'](x)
                q = layer['q'](x_norm)
                k = layer['k'](x_norm)
                v = layer['v'](x_norm)
                attn = F.scaled_dot_product_attention(q, k, v)
                attn = layer['out'](attn)
                x = x + attn
                x_norm2 = layer['norm2'](x)
                ffn = layer['fc1'](x_norm2)
                ffn = F.gelu(ffn)
                ffn = layer['fc2'](ffn)
                x = x + ffn
            x = self.norm(x)
            x = x.mean(dim=1)                              # average pooling
            return self.head(x)

    tiny = DKMCompressedSpectralGPT().to(DEVICE)

    with torch.no_grad():
        # Patch embedding
        if 'patch_embed' in original_model.compression_modules:
            mod = original_model.compression_modules['patch_embed']
            centroids = mod.get_centroids()
            orig_weight = original_model.patch_embed.weight.data
            dist = torch.cdist(orig_weight, centroids)
            assignments = dist.argmin(dim=1)
            reconstructed = centroids[assignments]
            K = mod.K
            tiny.patch_embed.weight.data = reconstructed[:K]
            tiny.patch_embed.bias.data = original_model.patch_embed.bias.data[:K]
        else:
            tiny.patch_embed.weight.data = original_model.patch_embed.weight.data
            tiny.patch_embed.bias.data = original_model.patch_embed.bias.data

        tiny.pos_embed.data = original_model.pos_embed.data[:, :, :new_embed_dim].clone()

        for i, orig_layer in enumerate(original_model.blocks):
            tiny_layer = tiny.layers[i]

            # Q, K, V — each looked up independently
            for proj_name, proj_attr in [('q', 'q_proj'), ('k', 'k_proj'), ('v', 'v_proj')]:
                cname = f'{proj_name}_{i}'
                if cname in original_model.compression_modules:
                    mod = original_model.compression_modules[cname]
                    centroids = mod.get_centroids()
                    orig_weight = getattr(orig_layer, proj_attr).weight.data
                    dist = torch.cdist(orig_weight, centroids)
                    assignments = dist.argmin(dim=1)
                    reconstructed = centroids[assignments]
                    K = mod.K
                    tiny_layer[proj_name].weight.data = reconstructed[:K, :new_embed_dim]
                    tiny_layer[proj_name].bias.data = getattr(orig_layer, proj_attr).bias.data[:K]
                else:
                    tiny_layer[proj_name].weight.data = getattr(orig_layer, proj_attr).weight.data
                    tiny_layer[proj_name].bias.data = getattr(orig_layer, proj_attr).bias.data

            # Attention output
            out_name = f'attn_out_{i}'
            qk_dim = min(
                learned_K.get(f'q_{i}', new_embed_dim),
                learned_K.get(f'k_{i}', new_embed_dim),
                learned_K.get(f'v_{i}', new_embed_dim),
            )
            if out_name in original_model.compression_modules:
                mod = original_model.compression_modules[out_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.out_proj.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                tiny_layer['out'].weight.data = reconstructed[:new_embed_dim, :qk_dim]
                tiny_layer['out'].bias.data = orig_layer.out_proj.bias.data[:new_embed_dim]
            else:
                tiny_layer['out'].weight.data = orig_layer.out_proj.weight.data
                tiny_layer['out'].bias.data = orig_layer.out_proj.bias.data

            # FC1
            fc1_name = f'fc1_{i}'
            if fc1_name in original_model.compression_modules:
                mod = original_model.compression_modules[fc1_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.fc1.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                K1 = mod.K
                tiny_layer['fc1'].weight.data = reconstructed[:K1, :new_embed_dim]
                tiny_layer['fc1'].bias.data = orig_layer.fc1.bias.data[:K1]
            else:
                tiny_layer['fc1'].weight.data = orig_layer.fc1.weight.data
                tiny_layer['fc1'].bias.data = orig_layer.fc1.bias.data

            # FC2
            fc2_name = f'fc2_{i}'
            ffn1_k = learned_K.get(f'fc1_{i}', new_embed_dim * 4)
            if fc2_name in original_model.compression_modules:
                mod = original_model.compression_modules[fc2_name]
                centroids = mod.get_centroids()
                orig_weight = orig_layer.fc2.weight.data
                dist = torch.cdist(orig_weight, centroids)
                assignments = dist.argmin(dim=1)
                reconstructed = centroids[assignments]
                tiny_layer['fc2'].weight.data = reconstructed[:new_embed_dim, :ffn1_k]
                tiny_layer['fc2'].bias.data = orig_layer.fc2.bias.data[:new_embed_dim]
            else:
                tiny_layer['fc2'].weight.data = orig_layer.fc2.weight.data
                tiny_layer['fc2'].bias.data = orig_layer.fc2.bias.data

        tiny.norm.weight.data = original_model.norm.weight.data[:new_embed_dim]
        tiny.norm.bias.data = original_model.norm.bias.data[:new_embed_dim]
        tiny.head.weight.data = original_model.head.weight.data[:, :new_embed_dim]
        tiny.head.bias.data = original_model.head.bias.data

    return tiny

# -------------------- Main Experiment --------------------
def run_experiment(data_path, dataset_name, config, ep_baseline=100, ep_stage2=50, ep_fine=80):
    print("\n" + "="*80)
    print(f"DATASET: {dataset_name}")
    print(f"PCA Components: {config['pca_components']}, Patch Size: {config['patch_size']}")
    print("="*80)

    X, y = load_dataset(data_path, dataset_name, config)
    X_pca = applyPCA(X, config['pca_components'])
    X_patches, y_labels = createImageCubes(X_pca, y, windowSize=config['patch_size'])
    n_classes = len(np.unique(y_labels))
    X_train, X_test, y_train, y_test = train_test_split(X_patches, y_labels, test_size=0.25, stratify=y_labels, random_state=SEED)
    S, _, L = X_train.shape[1], X_train.shape[2], X_train.shape[3]

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)),
        batch_size=64, shuffle=True)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(DEVICE)

    # ---- Baseline (large) ----
    print("\n--- Training Baseline (No Compression) ---")
    baseline = LargeCompressibleSpectralGPT(
        n_classes, S, L, patch_size=config['patch_size'], spectral_group=3,
        embed_dim=128, num_heads=8, num_layers=4,
        decoder_embed_dim=64, decoder_num_heads=4, decoder_depth=2,
        mlp_ratio=4.0, compress_layers=[]).to(DEVICE)
    baseline_start = time.time()
    baseline = train_two_stage(baseline, train_loader, X_test_t, y_test_t,
                               epochs_stage1=ep_baseline, epochs_stage2=0, final_lambda=0.0)
    baseline_train_time = time.time() - baseline_start
    baseline_metrics = evaluate_full(baseline, X_test_t, y_test_t)
    baseline_params = sum(p.numel() for p in baseline.parameters())
    baseline_size_mb = baseline_params * 4 / (1024 * 1024)

    # ---- Self-adaptive compressed model ----
    print("\n--- Training Self-Adaptive Compressed SpectralGPT (all layers) ---")
    compress_candidates = ['patch_embed']
    for i in range(4):
        compress_candidates.extend([
            f'q_{i}', f'k_{i}', f'v_{i}',
            f'attn_out_{i}', f'fc1_{i}', f'fc2_{i}'
        ])
    student = LargeCompressibleSpectralGPT(
        n_classes, S, L, patch_size=config['patch_size'], spectral_group=3,
        embed_dim=128, num_heads=8, num_layers=4,
        decoder_embed_dim=64, decoder_num_heads=4, decoder_depth=2,
        mlp_ratio=4.0, compress_layers=compress_candidates, gate_init=0.0).to(DEVICE)
    student_start = time.time()
    student = train_two_stage(student, train_loader, X_test_t, y_test_t,
                              epochs_stage1=120, epochs_stage2=ep_stage2, final_lambda=3.0, lr=0.0005)
    student_train_time = time.time() - student_start

    learned_K = student.get_learned_K()
    print("\n--- Learned K values ---")
    for k, v in learned_K.items():
        print(f"  {k}: {v}")

    # Build tiny model
    input_dim = S * S * L
    tiny_model = build_dkm_tiny_model(student, learned_K, input_dim, n_classes,
                                      num_layers=4, patch_size=config['patch_size'],
                                      spectral_group=3)
    tiny_params = sum(p.numel() for p in tiny_model.parameters())
    reduction = (1 - tiny_params / baseline_params) * 100
    print(f"\nTiny model parameters: {tiny_params:,} (reduction {reduction:.1f}% from baseline {baseline_params:,})")

    # Fine-tune tiny model
    print("\n--- Fine-tuning DKM Compressed Tiny SpectralGPT ---")
    opt_ft = optim.Adam(tiny_model.parameters(), lr=0.0001, weight_decay=1e-5)
    for epoch in range(ep_fine):
        tiny_model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt_ft.zero_grad()
            out = tiny_model(xb)
            loss = F.cross_entropy(out, yb)
            loss.backward()
            opt_ft.step()
            total_loss += loss.item()
        if (epoch+1) % 20 == 0:
            tiny_model.eval()
            with torch.no_grad():
                out_val = tiny_model(X_test_t)
                val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
            print(f"  FT Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}")

    tiny_metrics = evaluate_full(tiny_model, X_test_t, y_test_t)

    # Final comparison
    print("\n" + "="*80)
    print(f"FINAL RESULTS FOR {dataset_name}")
    print("="*80)
    print(f"{'Metric':<20} {'Baseline':<18} {'Tiny DKM Compressed':<18} {'Change':<12}")
    print("-"*70)
    print(f"{'Accuracy (%)':<20} {baseline_metrics['accuracy']*100:<18.2f} {tiny_metrics['accuracy']*100:<18.2f} {(tiny_metrics['accuracy']-baseline_metrics['accuracy'])*100:>+11.2f}%")
    print(f"{'Precision (%)':<20} {baseline_metrics['precision']*100:<18.2f} {tiny_metrics['precision']*100:<18.2f}")
    print(f"{'Recall (%)':<20} {baseline_metrics['recall']*100:<18.2f} {tiny_metrics['recall']*100:<18.2f}")
    print(f"{'F1 Score (%)':<20} {baseline_metrics['f1']*100:<18.2f} {tiny_metrics['f1']*100:<18.2f}")
    print(f"{'Parameters':<20} {baseline_params:<18,} {tiny_params:<18,} {reduction:>+11.1f}%")
    print(f"{'Model Size (MB)':<20} {baseline_size_mb:<18.2f} {tiny_params*4/(1024*1024):<18.2f} {((1-tiny_params/baseline_params)*100):>+11.1f}%")
    print(f"{'Inference Time (ms)':<20} {baseline_metrics['inf_time_ms']:<18.2f} {tiny_metrics['inf_time_ms']:<18.2f} {(tiny_metrics['inf_time_ms']/baseline_metrics['inf_time_ms']-1)*100:>+11.1f}%")
    print(f"{'Training Time (s)':<20} {baseline_train_time:<18.2f} {student_train_time:<18.2f} {(student_train_time/baseline_train_time-1)*100:>+11.1f}%")

Using device: cpu


In [2]:
config =  {
        'file': 'Botswana.mat',
        'gt_file': 'Botswana_gt.mat',
        'data_key': 'Botswana',
        'gt_key': 'Botswana_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Botswana Delta, 14 classes'
    }
data_path = r"Dataset"
dataset_name = 'Botswana'
run_experiment(data_path, dataset_name, config, 100, 120, 200)


DATASET: Botswana
PCA Components: 30, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0015 | Val Acc: 0.9951
Early stopping at epoch 38 (best val_acc=0.9951)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed SpectralGPT (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0018 | Val Acc: 0.9975
Early stopping at epoch 38 (best val_acc=0.9975)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/120 | λ=0.227 | Val Acc=1.0000
   Gates: {'patch_embed': 0.5181923508644104, 'q_0': 0.508941113948822, 'k_0': 0.5069200992584229, 'v_0': 0.5119447708129883, 'attn_out_0': 0.5138306021690369, 'fc1_0': 0.5072786211967468, 'fc2_0': 0.5128173232078552, 'q_1': 0.49847081303596497, 'k_1': 0.4962921738624573, 'v_1': 0.4978596866130829, 'attn_out_1': 0.49835941195487976, 'fc1_1': 0.5204377174377441, 'fc2_1':

In [3]:
config =  {
        'file': 'Indian_pines_corrected.mat',
        'gt_file': 'Indian_pines_gt.mat',
        'data_key': 'indian_pines_corrected',
        'gt_key': 'indian_pines_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Indian Pines, 16 classes'
    }
data_path = r"Dataset"
dataset_name = 'IndianPines'
run_experiment(data_path, dataset_name, config, 100, 20, 400)


DATASET: IndianPines
PCA Components: 30, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0085 | Val Acc: 0.9329
Stage1 Epoch 40/100 | Loss: 0.0002 | Val Acc: 0.9540
Stage1 Epoch 60/100 | Loss: 0.0001 | Val Acc: 0.9547
Early stopping at epoch 72 (best val_acc=0.9555)

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed SpectralGPT (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0117 | Val Acc: 0.9586
Stage1 Epoch 40/120 | Loss: 0.0002 | Val Acc: 0.9703
Stage1 Epoch 60/120 | Loss: 0.0001 | Val Acc: 0.9707
Stage1 Epoch 80/120 | Loss: 0.0030 | Val Acc: 0.9719
Stage1 Epoch 100/120 | Loss: 0.0001 | Val Acc: 0.9739
Stage1 Epoch 120/120 | Loss: 0.0001 | Val Acc: 0.9742

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 10/20 | λ=1.421 | Val Acc=0.9719
   Gates: {'patch_embed': 0.6045835614204407, 

In [2]:
config =  {
        'file': 'Salinas_corrected.mat',
        'gt_file': 'Salinas_gt.mat',
        'data_key': 'salinas_corrected',
        'gt_key': 'salinas_gt',
        'pca_components': 24,
        'patch_size': 5,
        'description': 'Salinas Valley, 16 classes'
    }
data_path = r"Dataset"
dataset_name = 'Salinas'
run_experiment(data_path, dataset_name, config, 100, 7, 200)


DATASET: Salinas
PCA Components: 24, Patch Size: 5

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0042 | Val Acc: 0.9950
Stage1 Epoch 40/100 | Loss: 0.0040 | Val Acc: 0.9919
Stage1 Epoch 60/100 | Loss: 0.0001 | Val Acc: 0.9976
Stage1 Epoch 80/100 | Loss: 0.0028 | Val Acc: 0.9970
Stage1 Epoch 100/100 | Loss: 0.0023 | Val Acc: 0.9964

--- Stage 2: Adding compression loss (K evolution) ---

--- Training Self-Adaptive Compressed SpectralGPT (all layers) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.0048 | Val Acc: 0.9962
Stage1 Epoch 60/120 | Loss: 0.0001 | Val Acc: 0.9974
Stage1 Epoch 80/120 | Loss: 0.0017 | Val Acc: 0.9974
Stage1 Epoch 100/120 | Loss: 0.0004 | Val Acc: 0.9988
Early stopping at epoch 104 (best val_acc=0.9994)

--- Stage 2: Adding compression loss (K evolution) ---

📊 Epoch 7/7 | λ=3.000 | Val Acc=0.9919
   Gates: {'patch_embed': 0.8302497863769531, 'q_0'